In [ ]:
import os  # OPENAI_API_KEY must be set in the environment

sample_text = """Machine learning is a subset of AI. It enables systems to learn from data.

Deep learning uses neural networks. These networks have many layers.

RAG combines retrieval with generation. It reduces hallucinations in LLMs."""

Fixed-Size (Character) Chunking
Cuts text into blocks of a fixed character count. It doesn't care about meaning or sentence boundaries 

In [ ]:
from langchain.text_splitter import CharacterTextSplitter

splitter = CharacterTextSplitter(
    separator=" ",        
    chunk_size=80,        
    chunk_overlap=20,     
    length_function=len,
)

chunks = splitter.split_text(sample_text)
for i, c in enumerate(chunks):
    print(f"Chunk {i}: {repr(c)}\n")

Chunk 0: 'Machine learning is a subset of AI. It enables systems to learn from data.\n\nDeep'

Chunk 1: 'from data.\n\nDeep learning uses neural networks. These networks have many'

Chunk 2: 'networks have many layers.\n\nRAG combines retrieval with generation. It reduces'

Chunk 3: 'It reduces hallucinations in LLMs.'



Recursive Character Chunking
Tries to split on the largest natural boundary first (paragraphs → \n\n), and only if a chunk is still too big does it fall back to smaller separators (lines → spaces → characters). This keeps related text together far better than fixed-size

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=80,
    chunk_overlap=20,
    separators=["\n\n", "\n", " ", ""],  
)

chunks = splitter.split_text(sample_text)
for i, c in enumerate(chunks):
    print(f"Chunk {i}: {repr(c)}\n")

Chunk 0: 'Machine learning is a subset of AI. It enables systems to learn from data.'

Chunk 1: 'Deep learning uses neural networks. These networks have many layers.'

Chunk 2: 'RAG combines retrieval with generation. It reduces hallucinations in LLMs.'



Token-Based Chunking
Measures size in tokens (the units an LLM actually reads) instead of characters. Since "chunk_size in characters" doesn't map cleanly to "what fits in the model's context window," token-based splitting gives you precise control over context budget.

In [ ]:
splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name="cl100k_base",  #GPT-3.5/4 tokenizer
    chunk_size=20,    #Token Count            
    chunk_overlap=5,
)

chunks = splitter.split_text(sample_text)
for i, c in enumerate(chunks):
    print(f"Chunk {i} (~tokens): {repr(c)}\n")

Chunk 0 (~tokens): 'Machine learning is a subset of AI. It enables systems to learn from data.'

Chunk 1 (~tokens): 'Deep learning uses neural networks. These networks have many layers.'

Chunk 2 (~tokens): 'RAG combines retrieval with generation. It reduces hallucinations in LLMs.'



Document-Specific Chunking
Splits according to the document's structure, Markdown headers, code functions, HTML tags — instead of blind text rules

In [6]:
from langchain.text_splitter import MarkdownHeaderTextSplitter

markdown_text = """# AI Basics
Machine learning learns from data.

## Deep Learning
Uses neural networks with many layers.

## RAG
Combines retrieval and generation."""

splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "H1"), ("##", "H2")]
)

chunks = splitter.split_text(markdown_text)
for c in chunks:
    print(f"Content: {c.page_content}")
    print(f"Metadata: {c.metadata}\n")

Content: Machine learning learns from data.
Metadata: {'H1': 'AI Basics'}

Content: Uses neural networks with many layers.
Metadata: {'H1': 'AI Basics', 'H2': 'Deep Learning'}

Content: Combines retrieval and generation.
Metadata: {'H1': 'AI Basics', 'H2': 'RAG'}



In [8]:
from langchain.text_splitter import  Language

python_code = """def greet(name):
    return f"Hello {name}"

def add(a, b):
    return a + b"""

splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=50, chunk_overlap=0
)
chunks = splitter.split_text(python_code)
for i, c in enumerate(chunks):
    print(f"Chunk {i}:\n{c}\n")

Chunk 0:
def greet(name):
    return f"Hello {name}"

Chunk 1:
def add(a, b):
    return a + b



Semantic Chunking
Instead of any fixed size, it embeds each sentence and measures how similar consecutive sentences are. When the topic shifts (similarity drops sharply), it cuts there. So chunks break at meaning boundaries.

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai import OpenAIEmbeddings

splitter = SemanticChunker(
    OpenAIEmbeddings(api_key= os.environ["OPENAI_API_KEY"] , 
                                  model="text-embedding-3-small"),
    breakpoint_threshold_type="percentile",  # cut at biggest meaning-jumps
    breakpoint_threshold_amount=90,
)

chunks = splitter.create_documents([sample_text])
for i, c in enumerate(chunks):
    print(f"Chunk {i}: {c.page_content}\n")

Chunk 0: Machine learning is a subset of AI. It enables systems to learn from data.

Chunk 1: Deep learning uses neural networks.

Chunk 2: These networks have many layers. RAG combines retrieval with generation. It reduces hallucinations in LLMs.



Parent-Document (Small-to-Big) Chunking
Solves a core tension, small chunks match queries precisely, but large chunks give the LLM better context. So it stores small "child" chunks for searching, but when one matches, it returns the larger "parent" chunk it came from.

In [ ]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

# Parent = bigger context, Child = smaller for precise matching
# KEY: chunk_overlap must be SMALLER than chunk_size for BOTH
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
child_splitter  = RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=20)

retriever = ParentDocumentRetriever(
    vectorstore=Chroma(
        collection_name="demo",
        embedding_function=OpenAIEmbeddings(api_key= os.environ["OPENAI_API_KEY"] , 
                                  model="text-embedding-3-small"),
    ),
    docstore=InMemoryStore(),
    child_splitter=child_splitter,
    parent_splitter=parent_splitter,
)

sample_text = """Machine learning is a subset of AI. It enables systems to learn from data.

Deep learning uses neural networks. These networks have many layers.

RAG combines retrieval with generation. It reduces hallucinations in LLMs."""

retriever.add_documents([Document(page_content=sample_text)])

results = retriever.invoke("What is RAG?")
print(results[0].page_content)

/var/folders/d_/47lp0l293sx5khzrnt9d5_2m0000gn/T/ipykernel_51626/4291052587.py:14: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the langchain-chroma package and should be used instead. To use it run `pip install -U langchain-chroma` and import as `from langchain_chroma import Chroma`.
  vectorstore=Chroma(


Machine learning is a subset of AI. It enables systems to learn from data.

Deep learning uses neural networks. These networks have many layers.

RAG combines retrieval with generation. It reduces hallucinations in LLMs.


In [4]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

llm = ChatOpenAI(api_key= os.environ["OPENAI_API_KEY"] , 
                 model="gpt-4o-mini", temperature=0)
embeddings = OpenAIEmbeddings(api_key= os.environ["OPENAI_API_KEY"] , 
                                  model="text-embedding-3-small")

docs = [
    Document(page_content="RAG combines retrieval with generation to reduce hallucinations."),
    Document(page_content="LangGraph builds stateful, multi-step agent workflows."),
    Document(page_content="Vector databases store embeddings for semantic search."),
]
vectorstore = Chroma.from_documents(docs, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 2})

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

prompt = ChatPromptTemplate.from_template(
    "Answer using only this context:\n{context}\n\nQuestion: {question}"
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(rag_chain.invoke("What is RAG?"))

RAG combines retrieval with generation to reduce hallucinations.


In [7]:
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever

bm25_retriever = BM25Retriever.from_documents(docs)   # keyword
bm25_retriever.k = 2
vector_retriever = vectorstore.as_retriever(search_kwargs={"k": 2})  # semantic

hybrid_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.4, 0.6],   # tune the balance
)

docs = hybrid_retriever.invoke("LangGraph workflows")
print(format_docs(docs))

LangGraph builds stateful, multi-step agent workflows.

Vector databases store embeddings for semantic search.


In [ ]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k": 2}),
    llm=llm,
)

docs = multi_query_retriever.invoke("What is RAG?")
print(f"Retrieved {len(docs)} unique docs")

Retrieved 2 unique docs


In [8]:
from langchain.retrievers import ContextualCompressionRetriever
from langchain.retrievers.document_compressors import CohereRerank

# Retrieve 10, rerank, keep top 3
base_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})
compressor = CohereRerank(model="rerank-english-v3.0", top_n=3)

reranking_retriever = ContextualCompressionRetriever(
    base_compressor=compressor,
    base_retriever=base_retriever,
)

rerank_chain = (
    {"context": reranking_retriever | format_docs, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)
print(rerank_chain.invoke("What is RAG?"))

/var/folders/d_/47lp0l293sx5khzrnt9d5_2m0000gn/T/ipykernel_65324/2998354773.py:6: LangChainDeprecationWarning: The class `CohereRerank` was deprecated in LangChain 0.0.30 and will be removed in 1.0. An updated version of the class exists in the langchain-cohere package and should be used instead. To use it run `pip install -U langchain-cohere` and import as `from langchain_cohere import CohereRerank`.
  compressor = CohereRerank(model="rerank-english-v3.0", top_n=3)


ImportError: Could not import cohere python package. Please install it with `pip install cohere`.

In [ ]:
Prompt Templ + Prompt + External Data = LLM = Response

In [ ]:
from langchain.tools.retriever import create_retriever_tool
from langgraph.prebuilt import create_react_agent

retriever_tool = create_retriever_tool(
    retriever,
    name="knowledge_base",
    description="Search docs about RAG, LangGraph, and vector databases.",
)

# Agent decides: use the tool? use it twice? answer directly?
agent = create_react_agent(llm, tools=[retriever_tool])

result = agent.invoke({
    "messages": [("user", "Compare RAG and LangGraph using the knowledge base.")]
})
print(result["messages"][-1].content)

In [ ]:
Step-1 : Prompt = LLM - Action - Shall I search for external Data? Yes or No - Call the External search retriever_tool
    Retrieved Info = LLM = Response

In [ ]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, List
from langchain_core.prompts import ChatPromptTemplate

class State(TypedDict):
    question: str
    documents: List[str]
    answer: str

def retrieve(state):
    docs = retriever.invoke(state["question"])
    return {"documents": [d.page_content for d in docs]}

def grade(state):
    """LLM grades whether docs are relevant."""
    grader = ChatPromptTemplate.from_template(
        "Docs: {docs}\nQuestion: {q}\nAre docs relevant? Answer 'yes' or 'no'."
    ) | llm | StrOutputParser()
    verdict = grader.invoke({"docs": state["documents"], "q": state["question"]})
    return {"answer": "RELEVANT" if "yes" in verdict.lower() else "NEEDS_WEB"}

def decide(state):
    return "generate" if state["answer"] == "RELEVANT" else "web_search"

def web_search(state):
    # Fallback: in production call a real search API (Tavily, etc.)
    return {"documents": state["documents"] + ["[web search results here]"]}

def generate(state):
    chain = ChatPromptTemplate.from_template(
        "Context: {docs}\nQuestion: {q}\nAnswer:"
    ) | llm | StrOutputParser()
    return {"answer": chain.invoke({"docs": state["documents"], "q": state["question"]})}

workflow = StateGraph(State)
workflow.add_node("retrieve", retrieve)
workflow.add_node("grade", grade)
workflow.add_node("web_search", web_search)
workflow.add_node("generate", generate)

workflow.set_entry_point("retrieve")
workflow.add_edge("retrieve", "grade")
workflow.add_conditional_edges("grade", decide,
    {"generate": "generate", "web_search": "web_search"})
workflow.add_edge("web_search", "generate")
workflow.add_edge("generate", END)

app = workflow.compile()
print(app.invoke({"question": "What is RAG?"})["answer"])